### CRISP-DM Phase 4.1 - Modeling : Zero-shot LLM classification

In [ ]:
import pandas as pd
import ast
import os
from dotenv import load_dotenv

from ollama import Client
from ollama import chat

from codecarbon import EmissionsTracker

In [ ]:
# Load the dataset
law = pd.read_csv('data/law.csv')
benchmark = pd.read_csv('data/benchmark.csv')

# Fix the format of the hazard column 
benchmark['Hazard'] = benchmark['Hazard'].apply(lambda x: x if isinstance(x, list) else ast.literal_eval(x))

In [ ]:
HAZARDS = ['flood', 'drought', 'temperature_extremes', 'sea_level_rise', 'storm', 'wildfire',
          'melting', 'erosion', 'other', 'none']

PROMPT = """Classify the following climate law or policy summary into one or more hazard categories.
- Only use labels from this list: {list}.
- Use 'none' only if the document has no relation to any physical climate hazard. 
- Use 'other' if the document addresses climate hazards not covered by the other categories.
- Assign all applicable labels.

Summary: {summary}"""

Ollama models (API)

In [ ]:
## Configure API
load_dotenv() 
ollama_client = Client(host='https://ollama.com', headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')})

def llm_classifier(text, model):
    if pd.isna(text) or text.strip() == '':
        return ['none']
    try:
        messages = [
            {
                'role': 'system',
                'content': 'You are an expert in climate policy classification. Only respond with a JSON array of hazard labels. Never include explanations.'
            },
            {
                'role': 'user',
                'content': PROMPT.format(summary=text, list=HAZARDS),
            },
        ]
        raw = ''
        for part in ollama_client.chat(model, messages=messages, stream=True):
            raw += part.message.content
        raw = raw.strip().replace('```json', '').replace('```', '').strip()
        labels = ast.literal_eval(raw)
        valid = [l for l in labels if l in HAZARDS]
        return valid if valid else ['none']
    
    except Exception as e:
        print(f"Error: {e}")
        return ['none']

In [ ]:
## Label benchmark documents with LLM classifier
models_dict = {'Hazard_gpt20': 'gpt-oss:20b', 'Hazard_gpt120': 'gpt-oss:120b',
               'Hazard_gemma4': 'gemma3:4b', 'Hazard_gemma27': 'gemma3:27b',
               'Hazard_minimax': 'minimax-m2.5', 'Hazard_minimax2': 'minimax-m2',
               'Hazard_qwen480': 'qwen3-coder:480b', 'Hazard_qwen80': 'qwen3-next:80b'}

for name, model in models_dict.items():
    tracker = EmissionsTracker()
    tracker.start()
    
    benchmark[name] = benchmark['Summary'].apply(lambda x: llm_classifier(x, model))

    emissions = tracker.stop()
    print(f"Emissions: {emissions} kg CO₂")

In [ ]:
## Export .csv
benchmark.to_csv('data/benchmark_llm.csv', index=False)